# 03 Q2: p53 ODE Model and PRISM Doxorubicin Response

This notebook answers Q2:

**Can the p53 ODE model predict response to a DNA-damaging drug, doxorubicin, in breast cancer cell lines?**

PRISM/DepMap provides baseline cell-line expression plus measured doxorubicin viability response. Here, p53 ODE features are tested against `doxorubicin_sensitivity_score`, where higher values mean greater doxorubicin sensitivity.

## Modelling Approach

The same adapted p53 DDR model used for Q1 is applied to the PRISM/DepMap breast cancer cell lines. Baseline expression is converted into gene-wise ratios relative to the PRISM/DepMap median before being used as multiplicative model inputs.

The main pre-specified p53 feature is again `p53_S46_AUC_across_DDR`, so Q1 and Q2 use a consistent mechanistic summary.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import odeint
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

prism_path = project_root / "data/processed/brca_prism_doxorubicin_modelling_table.csv"
scores_path = project_root / "data/processed/brca_prism_p53_ode_scores.csv"
correlation_results_path = project_root / "tables/q2_p53_ode_prism_correlation_results.csv"
summary_path = project_root / "tables/q2_p53_ode_prism_feature_summary.csv"
scatter_figure_path = project_root / "figures/q2_p53_ode_score_vs_prism_doxorubicin_sensitivity.png"
distribution_figure_path = project_root / "figures/q2_p53_ode_cellline_feature_distribution.png"

for folder in [scores_path.parent, correlation_results_path.parent, summary_path.parent, scatter_figure_path.parent, distribution_figure_path.parent]:
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:

# Minimal adapted p53 DDR ODE scorer from the course template.
# The input expression values are converted to gene-wise ratios relative to the dataset median.
# These ratios personalise the template's initial conditions or synthesis rates.

p53_model_genes = ["ATM", "CHEK2", "HIPK2", "MDM2", "PPM1D", "SIAH1", "TP53", "WSB1"]
ddr_values = np.linspace(0.01, 1.00, 10)
time_grid = np.linspace(0, 50, 10)

# Selected fitted parameter row ported from the course template parameter table.
# These names match the ODE equations below.
p53_params = {
    "kpatm_dox": 0.7141284576673614,
    "jpatm_dox": 0.01001501738522647,
    "kdpatm_dox": 4.287004777747307,
    "jdpatm_dox": 0.022712818746653787,
    "kpchk2": 39.99520814970714,
    "jpchk2": 0.10021818242876597,
    "kdpchk2": 15.246574360684299,
    "jdpchk2": 0.9995440018783482,
    "kpp53_ATM": 5.63137993400672,
    "kpp53_CHK2": 2.6323732318132933,
    "jpp53_ATM": 0.5414577662095134,
    "jpp53_CHK2": 0.583144727403126,
    "kdpp53a": 0.9757782419741889,
    "jdpp53a": 0.01044114413709567,
    "kpp53a": 35.63339377162273,
    "jpp53a": 0.9420163603515592,
    "kdpp53k": 0.7062534360373444,
    "jdpp53k": 0.19506211836488466,
    "kpsiah1": 0.30036132299082025,
    "jpsiah1": 0.01005366224642904,
    "kdpsiah1": 0.3001326978126625,
    "jdpsiah1": 0.8700047866744796,
    "ksmdm21": 0.6229893448940808,
    "jsmdm21": 0.10106525529238898,
    "kswip11": 0.6810544057024757,
    "jswip11": 0.20393136945804016,
    "kdp53": 2.000114586680669,
    "kdp53a": 0.10000649341988067,
    "jup53": 0.09933440059388587,
    "jup53s": 9.936632242874452,
    "juhipk2": 0.04007319402881099,
}

# Fixed constants and baseline initial conditions from the template.
hcmdm21 = 2
hcwip11 = 2
kdhipk2 = 1.5
kdp53m = 0.077
kdwip1m = 0.66
kdwip1 = 2
kdmdm2m = 0.7
kdmdm2 = 2
kuhipk2 = 0.17
hcdeg = 2
kalg = 0.1

p_p53i_0 = 0.1
p_MDM2m_0 = 0.103
p_p53m_0 = 0.1
WIP1m_0 = 0.0594
WIP1_0 = 0.0277
MDM2_0 = 1
p_ATM_0 = 1
p_SIAH_0 = 1
p_HIPK2_0 = 0.018
p_CHK2_0 = 1
ic_zero = 0

kpsmdm2_base = MDM2_0 * kdmdm2 / p_MDM2m_0
h_MDM2_0 = (MDM2_0 ** hcdeg) / (MDM2_0 ** hcdeg + p53_params["jup53"] ** hcdeg)
kpsp53_base = p53_params["kdp53"] * p_p53i_0 / p_p53m_0 * h_MDM2_0
kpswip1_base = kdwip1 * WIP1_0 / WIP1m_0
p_kshipk2_base = kdhipk2 * p_HIPK2_0 + kuhipk2 * p_SIAH_0 * p_HIPK2_0 / (p53_params["juhipk2"] + p_HIPK2_0)


def make_expression_ratios(data, genes):
    ratios = data[genes].astype(float).copy()
    for gene in genes:
        median_value = ratios[gene].median()
        ratios[gene] = ratios[gene] / median_value
    return ratios.clip(lower=0.05, upper=20)


def sample_initial_state_and_rates(row):
    ksp530 = row["TP53"] * p_p53m_0 * kdp53m

    MDM2m = row["MDM2"] * p_MDM2m_0
    MDM2 = MDM2m * kpsmdm2_base / kdmdm2
    h_MDM2 = (MDM2 ** hcdeg) / (MDM2 ** hcdeg + p53_params["jup53"] ** hcdeg)
    p53i = row["TP53"] * p_p53m_0 * kpsp53_base / p53_params["kdp53"] / h_MDM2
    h_p53i = ((kalg * p53i) ** hcmdm21) / (p53_params["jsmdm21"] ** hcmdm21 + (kalg * p53i) ** hcmdm21)
    ksmdm20 = MDM2m * kdmdm2m - p53_params["ksmdm21"] * h_p53i
    if ksmdm20 < 0:
        ksmdm20 = 1e-9

    WIP1m = row["PPM1D"] * WIP1m_0
    kswip10 = WIP1m * kdwip1m

    SIAH_0 = 0.5 * (row["SIAH1"] + row["WSB1"]) * p_SIAH_0
    kshipk2 = row["HIPK2"] * p_kshipk2_base
    hipk2_term = (
        SIAH_0 ** 2 * kuhipk2 ** 2
        + 2 * SIAH_0 * p53_params["juhipk2"] * kdhipk2 * kuhipk2
        - 2 * SIAH_0 * kshipk2 * kuhipk2
        + p53_params["juhipk2"] ** 2 * kdhipk2 ** 2
        + 2 * p53_params["juhipk2"] * kdhipk2 * kshipk2
        + kshipk2 ** 2
    )
    HIPK2 = (kshipk2 - SIAH_0 * kuhipk2 - p53_params["juhipk2"] * kdhipk2 + np.sqrt(max(hipk2_term, 0))) / (2 * kdhipk2)
    if HIPK2 < 0:
        HIPK2 = 1e-9

    x0 = [
        row["ATM"] * p_ATM_0,
        ic_zero,
        row["TP53"] * p_p53m_0,
        p53i,
        ic_zero,
        ic_zero,
        SIAH_0,
        ic_zero,
        HIPK2,
        WIP1m,
        WIP1m * kpswip1_base / kdwip1,
        row["CHEK2"] * p_CHK2_0,
        ic_zero,
        MDM2m,
        MDM2,
    ]
    rates = {
        "ksp530": ksp530,
        "kpsmdm2": kpsmdm2_base,
        "kpsp53": kpsp53_base,
        "kpswip1": kpswip1_base,
        "kshipk2": kshipk2,
        "kswip10": kswip10,
        "ksmdm20": ksmdm20,
    }
    return x0, rates


def p53_ode(x, t, ddr, rates):
    WIP1a = 1

    ATM, ATMp, p53m, p53i, p53s15, p53s46, SIAH1, SIAH1p, HIPK2, WIP1m, WIP1, CHK2, CHK2p, MDM2m, MDM2 = x
    ku = np.heaviside(t, 1)

    v1 = p53_params["kpatm_dox"] * ATM * ddr * ku / (p53_params["jpatm_dox"] + ATM)
    v2 = p53_params["kdpatm_dox"] * WIP1a * ATMp / (p53_params["jdpatm_dox"] + ATMp)
    v3 = p53_params["kpchk2"] * ATMp * CHK2 / (p53_params["jpchk2"] + CHK2)
    v4 = p53_params["kdpchk2"] * WIP1 * CHK2p / (p53_params["jdpchk2"] + CHK2p)
    v5 = p53_params["kpp53_CHK2"] * CHK2p * p53i / (p53_params["jpp53_CHK2"] + p53i)
    v6 = p53_params["kpp53_ATM"] * ATMp * p53i / (p53_params["jpp53_ATM"] + p53i)
    v7 = p53_params["kdpp53a"] * WIP1 * p53s15 / (p53_params["jdpp53a"] + p53s15)
    v8 = p53_params["kpp53a"] * HIPK2 * p53s15 / (p53_params["jpp53a"] + p53s15)
    v9 = p53_params["kdpp53k"] * p53s46 / (p53_params["jdpp53k"] + p53s46)
    v10 = p53_params["kpsiah1"] * ATMp * SIAH1 / (p53_params["jpsiah1"] + SIAH1)
    v11 = p53_params["kdpsiah1"] * SIAH1p / (p53_params["jdpsiah1"] + SIAH1p)

    p53alg = kalg * p53i + p53s15 + p53s46
    p53s15tot = p53s15 + p53s46
    v12 = rates["ksp530"]
    v13 = rates["ksmdm20"]
    v14 = p53_params["ksmdm21"] * (p53alg ** hcmdm21) / (p53_params["jsmdm21"] ** hcmdm21 + p53alg ** hcmdm21)
    v15 = rates["kswip10"]
    v16 = p53_params["kswip11"] * (p53s15tot ** hcwip11) / (p53_params["jswip11"] ** hcwip11 + p53s15tot ** hcwip11)

    v21 = rates["kpsp53"] * p53m
    v22 = rates["kshipk2"]
    v23 = rates["kpswip1"] * WIP1m
    v24 = rates["kpsmdm2"] * MDM2m

    v25 = p53_params["kdp53"] * p53i * (MDM2 ** hcdeg) / (MDM2 ** hcdeg + p53_params["jup53"] ** hcdeg)
    v26 = p53_params["kdp53a"] * p53s15 * (MDM2 ** hcdeg) / (MDM2 ** hcdeg + p53_params["jup53s"] ** hcdeg)
    v27 = p53_params["kdp53a"] * p53s46 * (MDM2 ** hcdeg) / (MDM2 ** hcdeg + p53_params["jup53s"] ** hcdeg)
    v28 = kdhipk2 * HIPK2
    v29 = kdp53m * p53m
    v30 = kdwip1m * WIP1m
    v31 = kdwip1 * WIP1
    v32 = kdmdm2m * MDM2m
    v33 = kdmdm2 * MDM2
    v36 = kuhipk2 * SIAH1 * HIPK2 / (p53_params["juhipk2"] + HIPK2)

    return [
        -v1 + v2,
        v1 - v2,
        v12 - v29,
        -v5 - v6 + v7 + v21 - v25,
        v5 + v6 - v7 - v8 + v9 - v26,
        v8 - v9 - v27,
        -v10 + v11,
        v10 - v11,
        v22 - v28 - v36,
        v15 + v16 - v30,
        v23 - v31,
        -v3 + v4,
        v3 - v4,
        v13 + v14 - v32,
        v24 - v33,
    ]


def simulate_p53_scores(data, id_columns):
    ratios = make_expression_ratios(data, p53_model_genes)
    records = []

    for index, row in ratios.iterrows():
        x0, rates = sample_initial_state_and_rates(row)
        s15_values = []
        s46_values = []

        for ddr in ddr_values:
            solution = odeint(p53_ode, x0, time_grid, args=(ddr, rates))
            s15_values.append(solution[-1, 4])
            s46_values.append(solution[-1, 5])

        record = {col: data.loc[index, col] for col in id_columns if col in data.columns}
        record["p53_S15_low_DDR"] = s15_values[0]
        record["p53_S15_high_DDR"] = s15_values[-1]
        record["p53_S15_AUC_across_DDR"] = np.trapezoid(s15_values, ddr_values)
        record["p53_S46_low_DDR"] = s46_values[0]
        record["p53_S46_high_DDR"] = s46_values[-1]
        record["p53_S46_AUC_across_DDR"] = np.trapezoid(s46_values, ddr_values)
        record["p53_S46_to_S15_ratio"] = record["p53_S46_AUC_across_DDR"] / (record["p53_S15_AUC_across_DDR"] + 1e-9)
        records.append(record)

    return pd.DataFrame(records)


## Load PRISM/DepMap Doxorubicin Table

The final PRISM/DepMap breast cancer modelling table has 26 cell lines. This small sample size is a major limitation for Q2.

In [ ]:
prism = pd.read_csv(prism_path)
required_columns = ["depmap_id", "cell_line_name", "prism_doxorubicin_lfc", "relative_abundance_percent", "doxorubicin_sensitivity_score"] + p53_model_genes
missing_columns = [col for col in required_columns if col not in prism.columns]

print("PRISM/DepMap table shape:", prism.shape)
print("Missing required columns:", missing_columns)
if missing_columns:
    raise ValueError("Missing required PRISM/DepMap columns for Q2 p53 ODE analysis.")

prism_q2 = prism.dropna(subset=required_columns).copy()
print("Cell lines available for Q2:", len(prism_q2))
print("Higher doxorubicin_sensitivity_score means greater doxorubicin sensitivity.")
display(prism_q2[["depmap_id", "cell_line_name", "prism_doxorubicin_lfc", "doxorubicin_sensitivity_score"] + p53_model_genes[:4]].head())

## Generate PRISM/DepMap p53 ODE Scores

Each cell line is scored by simulating the p53 DDR model across the same DDR dose grid as Q1.

In [ ]:
prism_scores = simulate_p53_scores(prism_q2, id_columns=["depmap_id", "cell_line_name"])
prism_scores.to_csv(scores_path, index=False)

print("Saved PRISM/DepMap p53 ODE scores to:", scores_path.relative_to(project_root))
print("Score table shape:", prism_scores.shape)
display(prism_scores.head())

In [ ]:
p53_feature_cols = [
    "p53_S15_low_DDR",
    "p53_S15_high_DDR",
    "p53_S15_AUC_across_DDR",
    "p53_S46_low_DDR",
    "p53_S46_high_DDR",
    "p53_S46_AUC_across_DDR",
    "p53_S46_to_S15_ratio",
]
main_p53_feature = "p53_S46_AUC_across_DDR"

feature_summary = prism_scores[p53_feature_cols].describe().T.reset_index().rename(columns={"index": "feature"})
feature_summary.to_csv(summary_path, index=False)
print("Saved Q2 feature summary to:", summary_path.relative_to(project_root))
display(feature_summary)

## Q2 Association With PRISM Doxorubicin Sensitivity

For each p53 feature, we calculate Spearman and Pearson correlations with `doxorubicin_sensitivity_score`.

For the main feature, we also fit a simple one-feature linear regression. This is descriptive only because n=26 is too small for strong predictive claims.

In [ ]:
q2_data = prism_q2[["depmap_id", "cell_line_name", "prism_doxorubicin_lfc", "relative_abundance_percent", "doxorubicin_sensitivity_score"]].merge(
    prism_scores[["depmap_id"] + p53_feature_cols],
    on="depmap_id",
    how="inner",
)

q2_rows = []
for feature in p53_feature_cols:
    feature_data = q2_data[[feature, "doxorubicin_sensitivity_score"]].dropna().copy()
    spearman_r, spearman_p = spearmanr(feature_data[feature], feature_data["doxorubicin_sensitivity_score"])
    pearson_r, pearson_p = pearsonr(feature_data[feature], feature_data["doxorubicin_sensitivity_score"])
    q2_rows.append({
        "feature": feature,
        "method": "spearman",
        "correlation": spearman_r,
        "p_value": spearman_p,
        "n_cell_lines": len(feature_data),
    })
    q2_rows.append({
        "feature": feature,
        "method": "pearson",
        "correlation": pearson_r,
        "p_value": pearson_p,
        "n_cell_lines": len(feature_data),
    })

X_main = q2_data[[main_p53_feature]].astype(float)
y_main = q2_data["doxorubicin_sensitivity_score"].astype(float)
linear_model = LinearRegression()
linear_model.fit(X_main, y_main)
linear_pred = linear_model.predict(X_main)
q2_rows.append({
    "feature": main_p53_feature,
    "method": "linear_regression_r2",
    "correlation": r2_score(y_main, linear_pred),
    "p_value": np.nan,
    "n_cell_lines": len(q2_data),
})
q2_rows.append({
    "feature": main_p53_feature,
    "method": "linear_regression_rmse",
    "correlation": np.sqrt(mean_squared_error(y_main, linear_pred)),
    "p_value": np.nan,
    "n_cell_lines": len(q2_data),
})

q2_results = pd.DataFrame(q2_rows)
q2_results.to_csv(correlation_results_path, index=False)
print("Saved Q2 PRISM correlation results to:", correlation_results_path.relative_to(project_root))
print("Main-feature linear regression slope:", float(linear_model.coef_[0]))
print("Main-feature linear regression intercept:", float(linear_model.intercept_))
display(q2_results)

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.hist(q2_data[main_p53_feature].dropna(), bins=12, edgecolor="black", color="#4c78a8")
plt.xlabel(main_p53_feature)
plt.ylabel("Number of breast cancer cell lines")
plt.title("Distribution of PRISM/DepMap p53 S46 DDR response score")
plt.tight_layout()
plt.savefig(distribution_figure_path, dpi=200)
plt.show()
print("Saved Q2 feature distribution figure to:", distribution_figure_path.relative_to(project_root))

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(q2_data[main_p53_feature], q2_data["doxorubicin_sensitivity_score"], color="#4c78a8", edgecolor="black", alpha=0.85)

line_x = np.linspace(q2_data[main_p53_feature].min(), q2_data[main_p53_feature].max(), 100)
line_y = linear_model.predict(pd.DataFrame({main_p53_feature: line_x}))
plt.plot(line_x, line_y, color="black", linestyle="--", linewidth=1)

main_spearman = q2_results[(q2_results["feature"] == main_p53_feature) & (q2_results["method"] == "spearman")].iloc[0]
main_pearson = q2_results[(q2_results["feature"] == main_p53_feature) & (q2_results["method"] == "pearson")].iloc[0]

plt.xlabel(main_p53_feature)
plt.ylabel("PRISM doxorubicin sensitivity score")
plt.title("p53 ODE score vs PRISM doxorubicin sensitivity")
plt.text(
    0.03,
    0.97,
    f"n = {len(q2_data)}\nSpearman r = {main_spearman['correlation']:.2f}\nPearson r = {main_pearson['correlation']:.2f}",
    transform=plt.gca().transAxes,
    va="top",
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "lightgray"},
)
plt.tight_layout()
plt.savefig(scatter_figure_path, dpi=200)
plt.show()
print("Saved Q2 scatter figure to:", scatter_figure_path.relative_to(project_root))

## Q2 Interpretation

Use the saved correlation table and scatterplot to report whether the main p53 ODE feature is associated with PRISM doxorubicin sensitivity.

A positive correlation means higher simulated p53 S46 DDR response is associated with greater doxorubicin sensitivity. A negative correlation means higher simulated p53 S46 DDR response is associated with lower doxorubicin sensitivity.

This should be interpreted cautiously because the final PRISM/DepMap breast cancer table contains only 26 cell lines.